After completing multiple initial models, we would like to create a master dataset for us to run all of our models against. This dataset will be the basis of all of our models from which we can later take a subsection from. The models we would like to run are logistic regression (lasso, ridge regression, and elastic net), KNN, and XG Boost. Below is our preprocessing steps to create this dataset. 

In [98]:
from pathlib import Path
import os

# Run the notebook from the root of your Formula-1 project.
PROJECT_ROOT = Path.cwd().resolve()

# Change this if your spark-warehouse folder is somewhere else.
WAREHOUSE_DIR = PROJECT_ROOT / "spark-warehouse"

print("Project root:", PROJECT_ROOT)
print("Spark warehouse:", WAREHOUSE_DIR)
print("Warehouse exists:", WAREHOUSE_DIR.exists())

Project root: /Users/alliewandling/Documents/uva_masters/BigData/Formula-1
Spark warehouse: /Users/alliewandling/Documents/uva_masters/BigData/Formula-1/spark-warehouse
Warehouse exists: True


In [99]:
from pathlib import Path

DATABASE_NAME = "formula1"

PROJECT_ROOT = Path.cwd().resolve()
WAREHOUSE_DIRECTORY = PROJECT_ROOT / "spark-warehouse"
DATABASE_DIRECTORY = WAREHOUSE_DIRECTORY / "formula1.db"

print("Current directory:", PROJECT_ROOT)
print("Warehouse:", WAREHOUSE_DIRECTORY)
print("Database:", DATABASE_DIRECTORY)
print("Database exists:", DATABASE_DIRECTORY.exists())

Current directory: /Users/alliewandling/Documents/uva_masters/BigData/Formula-1
Warehouse: /Users/alliewandling/Documents/uva_masters/BigData/Formula-1/spark-warehouse
Database: /Users/alliewandling/Documents/uva_masters/BigData/Formula-1/spark-warehouse/formula1.db
Database exists: True


In [100]:
if not DATABASE_DIRECTORY.exists():
    raise FileNotFoundError(
        f"Database directory not found: {DATABASE_DIRECTORY}"
    )

table_directories = sorted(
    path
    for path in DATABASE_DIRECTORY.iterdir()
    if path.is_dir()
)

for table_directory in table_directories:
    print(table_directory.name)

driver_race_modeling
driver_race_modeling_with_ratings
drivers
event
laps
results
session_info
session_status
track_status
weather_data


In [101]:
from pyspark.sql import SparkSession
import os

os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17" 

spark = SparkSession.builder \
    .appName("YourAppName") \
    .getOrCreate()

spark.sql(
    f"""
    CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}
    LOCATION '{DATABASE_DIRECTORY.as_uri()}'
    """
)

spark.sql(f"USE {DATABASE_NAME}")

spark.sql("SELECT current_database()").show()

+----------------+
|current_schema()|
+----------------+
|        formula1|
+----------------+



In [102]:
for table_directory in table_directories:
    table_name = table_directory.name

    if table_name.startswith((".", "_")):
        continue

    storage_format = (
        "DELTA"
        if (table_directory / "_delta_log").exists()
        else "PARQUET"
    )

    try:
        spark.sql(
            f"""
            CREATE TABLE IF NOT EXISTS
                {DATABASE_NAME}.`{table_name}`
            USING {storage_format}
            LOCATION '{table_directory.as_uri()}'
            """
        )

        print(
            f"Registered {DATABASE_NAME}.{table_name} "
            f"using {storage_format}"
        )

    except Exception as error:
        print(f"Could not register {table_name}: {error}")

Registered formula1.driver_race_modeling using PARQUET
Registered formula1.driver_race_modeling_with_ratings using PARQUET
Registered formula1.drivers using PARQUET
Registered formula1.event using PARQUET
Registered formula1.laps using PARQUET
Registered formula1.results using PARQUET
Registered formula1.session_info using PARQUET
Registered formula1.session_status using PARQUET
Registered formula1.track_status using PARQUET
Registered formula1.weather_data using PARQUET


In [103]:
spark.sql("SHOW TABLES IN formula1").show(truncate=False)

+---------+---------------------------------+-----------+
|namespace|tableName                        |isTemporary|
+---------+---------------------------------+-----------+
|formula1 |driver_race_modeling             |false      |
|formula1 |driver_race_modeling_with_ratings|false      |
|formula1 |drivers                          |false      |
|formula1 |event                            |false      |
|formula1 |laps                             |false      |
|formula1 |results                          |false      |
|formula1 |session_info                     |false      |
|formula1 |session_status                   |false      |
|formula1 |track_status                     |false      |
|formula1 |weather_data                     |false      |
|         |best_pit_spark                   |true       |
|         |driver_history                   |true       |
|         |event_info                       |true       |
|         |fastest_lap_spark                |true       |
|         |lap

In [104]:
DATABASE_NAME = "formula1"

if spark.catalog.databaseExists(DATABASE_NAME):
    spark.sql(f"USE {DATABASE_NAME}")
    print(f"Connected to database: {DATABASE_NAME}")
else:
    print(f"Database '{DATABASE_NAME}' was not found.")

Connected to database: formula1


In [105]:
DATABASE_NAME = "formula1"
DATABASE_PATH = WAREHOUSE_DIR / f"{DATABASE_NAME}.db"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}")
spark.sql(f"USE {DATABASE_NAME}")

required_tables = [
    "results",
    "laps",
    "weather_data",
    "drivers",
    "session_info",
    "session_status",
    "track_status"
]

for table_name in required_tables:
    table_path = DATABASE_PATH / table_name

    if table_path.exists():
        spark.sql(
            f"""
            CREATE TABLE IF NOT EXISTS {DATABASE_NAME}.{table_name}
            USING PARQUET
            LOCATION '{table_path.as_uri()}'
            """
        )

        print(f"Connected: {DATABASE_NAME}.{table_name}")
    else:
        print(f"Not found: {table_path}")

Connected: formula1.results
Connected: formula1.laps
Connected: formula1.weather_data
Connected: formula1.drivers
Connected: formula1.session_info
Connected: formula1.session_status
Connected: formula1.track_status


In [106]:
required_model_tables = [
    "results",
    "laps",
    "weather_data"
]

missing_tables = [
    table_name
    for table_name in required_model_tables
    if not spark.catalog.tableExists(
        f"formula1.{table_name}"
    )
]

if missing_tables:
    raise RuntimeError(
        "Missing required tables: "
        + ", ".join(missing_tables)
    )

print("All required tables are available.")

All required tables are available.


Now we will pull in the event information, so that we can use the Location as the race identifier instead of the session number. 

In [107]:
# Create the temporary view with a quoted string literal
spark.sql("""
    CREATE OR REPLACE TEMP VIEW event_info AS 
    SELECT * from formula1.event WHERE type_of_race = 'R'
""")

# Show the results
spark.sql("SELECT * FROM event_info").show(5)


+--------------------+----+--------------+------------+-----------+-----------+
|         source_file|year|session_number|type_of_race|RoundNumber|   Location|
+--------------------+----+--------------+------------+-----------+-----------+
|file:///Users/all...|2020|            10|           R|         10|      Sochi|
|file:///Users/all...|2020|            11|           R|         11|Nürburgring|
|file:///Users/all...|2020|            12|           R|         12|   Portimão|
|file:///Users/all...|2020|            13|           R|         13|      Imola|
|file:///Users/all...|2020|            14|           R|         14|   Istanbul|
+--------------------+----+--------------+------------+-----------+-----------+
only showing top 5 rows


In [108]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW results_clean AS

SELECT
    CAST(year AS INT) AS year,
    CAST(session_number AS INT) AS session_number,
    UPPER(TRIM(CAST(type_of_race AS STRING))) AS session_type,

    CAST(DriverNumber AS STRING) AS driver_number,
    CAST(FullName AS STRING) AS driver_name,
    NULLIF(TRIM(CAST(TeamName AS STRING)), '') AS team_name,

    TRY_CAST(Position AS INT) AS result_position,
    TRY_CAST(GridPosition AS INT) AS grid_position,
    TRY_CAST(Points AS DOUBLE) AS championship_points,

    CAST(Status AS STRING) AS status

FROM formula1.results
""")
spark.sql("SELECT * FROM results_clean").show(5, truncate=False)

+----+--------------+------------+-------------+----------------+---------------+---------------+-------------+-------------------+------+
|year|session_number|session_type|driver_number|driver_name     |team_name      |result_position|grid_position|championship_points|status|
+----+--------------+------------+-------------+----------------+---------------+---------------+-------------+-------------------+------+
|2021|12            |Q           |33           |Max Verstappen  |Red Bull Racing|1              |NULL         |NULL               |NULL  |
|2021|12            |Q           |63           |George Russell  |Williams       |2              |NULL         |NULL               |NULL  |
|2021|12            |Q           |44           |Lewis Hamilton  |Mercedes       |3              |NULL         |NULL               |NULL  |
|2021|12            |Q           |3            |Daniel Ricciardo|McLaren        |4              |NULL         |NULL               |NULL  |
|2021|12            |Q     

The results_clean table above shrinks the full results table to only include the columns year, session_number, session_type, driver_number, driver_id, team_name, result_position, grid_position, championship_points, and status. The driver_id is created from the driver abbreviation, which will be the main way to identify a different driver as dirver_number can change if they accept driver_number 1 after winning the season. Below we can see the breakdown of sessions by year.

In [109]:
spark.sql("""
SELECT
    year,
    session_type,
    COUNT(*) AS rows,
    COUNT(DISTINCT session_number) AS sessions
FROM results_clean
GROUP BY year, session_type
ORDER BY year, session_type
""").show(100)

+----+------------+----+--------+
|year|session_type|rows|sessions|
+----+------------+----+--------+
|2018|           Q| 420|      21|
|2019|           Q| 420|      21|
|2020|           Q| 340|      17|
|2020|           R| 340|      17|
|2021|           Q| 440|      22|
|2021|           R| 440|      22|
|2022|           Q| 440|      22|
|2022|           R| 440|      22|
|2023|           Q|  60|       3|
|2023|           R| 440|      22|
|2024|           R| 479|      24|
|2025|           Q| 480|      24|
|2025|           R| 479|      24|
|2026|           Q| 154|       7|
|2026|           R| 154|       7|
+----+------------+----+--------+



In [110]:
race_outcomes = spark.sql("""
SELECT
    year,
    session_number,
    driver_number,
    driver_name,
    team_name,

    result_position AS target_position,
    championship_points AS target_championship_points,

    grid_position,
    status,

    CASE
        WHEN grid_position > 0
         AND result_position IS NOT NULL
        THEN grid_position - result_position
        ELSE NULL
    END AS positions_gained,

    CASE
        WHEN status IS NULL THEN NULL
        WHEN LOWER(status) = 'finished' THEN 0
        WHEN LOWER(status) LIKE '+%lap%' THEN 0
        ELSE 1
    END AS dnf_flag

FROM results_clean

WHERE session_type = 'R'
  AND result_position IS NOT NULL
""")
race_outcomes.show(5)

+----+--------------+-------------+---------------+-----------+---------------+--------------------------+-------------+--------+----------------+--------+
|year|session_number|driver_number|    driver_name|  team_name|target_position|target_championship_points|grid_position|  status|positions_gained|dnf_flag|
+----+--------------+-------------+---------------+-----------+---------------+--------------------------+-------------+--------+----------------+--------+
|2025|             9|           81|  Oscar Piastri|    McLaren|              1|                      25.0|            1|Finished|               0|       0|
|2025|             9|            4|   Lando Norris|    McLaren|              2|                      18.0|            2|Finished|               0|       0|
|2025|             9|           16|Charles Leclerc|    Ferrari|              3|                      15.0|            7|Finished|               4|       0|
|2025|             9|           63| George Russell|   Mercedes| 

Race outcomes dataframe provides information based on driver_name, session_number and year for what each driver did. In the master dataset, we will have the session_number changed to Location as the session changes based on the calendar every year.

In [111]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW qualifying_results AS

WITH qualifying AS (
    SELECT
        year,
        session_number,
        driver_number,
        driver_name,
        team_name,
        result_position AS qualifying_position
    FROM results_clean
    WHERE session_type = 'Q'
      AND result_position IS NOT NULL
),

team_windows AS (
    SELECT
        *,

        COUNT(*) OVER (
            PARTITION BY year, session_number
        ) AS qualifying_field_size,

        COUNT(*) OVER (
            PARTITION BY year, session_number, team_name
        ) AS team_driver_count,

        SUM(qualifying_position) OVER (
            PARTITION BY year, session_number, team_name
        ) AS team_position_sum

    FROM qualifying
)

SELECT
    year,
    session_number,
    driver_number,
    driver_name,
    team_name,

    qualifying_position,
    qualifying_field_size,

    CASE
        WHEN qualifying_field_size > 1
        THEN
            (qualifying_position - 1.0)
            / (qualifying_field_size - 1.0)
    END AS qualifying_position_percentile,

    CASE
        WHEN qualifying_position <= 10 THEN 1
        ELSE 0
    END AS qualified_top_10,

    CASE
        WHEN team_driver_count = 2
        THEN team_position_sum - qualifying_position
    END AS teammate_qualifying_position,

    CASE
        WHEN team_driver_count = 2
        THEN
            qualifying_position
            - (team_position_sum - qualifying_position)
    END AS qualifying_vs_teammate

FROM team_windows
""")

spark.sql("SELECT * FROM qualifying_results""").show(5)

+----+--------------+-------------+----------------+------------+-------------------+---------------------+------------------------------+----------------+----------------------------+----------------------+
|year|session_number|driver_number|     driver_name|   team_name|qualifying_position|qualifying_field_size|qualifying_position_percentile|qualified_top_10|teammate_qualifying_position|qualifying_vs_teammate|
+----+--------------+-------------+----------------+------------+-------------------+---------------------+------------------------------+----------------+----------------------------+----------------------+
|2018|             1|            7|  Kimi Räikkönen|     Ferrari|                  2|                   20|          0.052631578947368...|               1|                           3|                    -1|
|2018|             1|            5|Sebastian Vettel|     Ferrari|                  3|                   20|          0.105263157894736...|               1|             

The qualifying dataframe contains information for the year, session, and driver_name for qualifying results. Qualifing position percentile informs us of where in the field the person qualified and the teammate qualifying information provides insight on how a person with a similar car performed. 

In [112]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW laps_clean AS

SELECT
    CAST(year AS INT) AS year,
    CAST(session_number AS INT) AS session_number,
    UPPER(TRIM(CAST(type_of_race AS STRING))) AS session_type,

    CAST(DriverNumber AS STRING) AS driver_number,
    CAST(Driver AS STRING) AS driver_abbreviation,
    CAST(Team AS STRING) AS team_name,

    TRY_CAST(LapNumber AS INT) AS lap_number,
    TRY_CAST(Position AS DOUBLE) AS running_position,
    TRY_CAST(Stint AS INT) AS stint,

    CAST(Compound AS STRING) AS compound,

    TRY_CAST(SpeedI1 AS DOUBLE) AS speed_i1,
    TRY_CAST(SpeedI2 AS DOUBLE) AS speed_i2,
    TRY_CAST(SpeedFL AS DOUBLE) AS finish_line_speed,
    TRY_CAST(SpeedST AS DOUBLE) AS speed_trap,

    CASE
        WHEN LapTime IS NULL
          OR LOWER(TRIM(CAST(LapTime AS STRING)))
             IN ('', 'nat', 'null', 'none')
        THEN NULL

        ELSE
            COALESCE(
                TRY_CAST(
                    NULLIF(
                        REGEXP_EXTRACT(
                            CAST(LapTime AS STRING),
                            '([0-9]+) days',
                            1
                        ),
                        ''
                    ) AS DOUBLE
                ),
                0
            ) * 86400

            +

            COALESCE(
                TRY_CAST(
                    NULLIF(
                        REGEXP_EXTRACT(
                            CAST(LapTime AS STRING),
                            'days ([0-9]{2}):',
                            1
                        ),
                        ''
                    ) AS DOUBLE
                ),
                0
            ) * 3600

            +

            COALESCE(
                TRY_CAST(
                    NULLIF(
                        REGEXP_EXTRACT(
                            CAST(LapTime AS STRING),
                            'days [0-9]{2}:([0-9]{2}):',
                            1
                        ),
                        ''
                    ) AS DOUBLE
                ),
                0
            ) * 60

            +

            COALESCE(
                TRY_CAST(
                    NULLIF(
                        REGEXP_EXTRACT(
                            CAST(LapTime AS STRING),
                            'days [0-9]{2}:[0-9]{2}:([0-9]+[.]?[0-9]*)',
                            1
                        ),
                        ''
                    ) AS DOUBLE
                ),
                0
            )
    END AS lap_time_seconds,
    CASE
        WHEN PitOutTime IS NULL
          OR LOWER(TRIM(CAST(PitOutTime AS STRING)))
             IN ('', 'nat', 'null', 'none')
        THEN 0

        ELSE
            COALESCE(
                TRY_CAST(
                    NULLIF(
                        REGEXP_EXTRACT(
                            CAST(PitOutTime AS STRING),
                            '([0-9]+) days',
                            1
                        ),
                        ''
                    ) AS DOUBLE
                ),
                0
            ) * 86400

            +

            COALESCE(
                TRY_CAST(
                    NULLIF(
                        REGEXP_EXTRACT(
                            CAST(PitOutTime AS STRING),
                            'days ([0-9]{2}):',
                            1
                        ),
                        ''
                    ) AS DOUBLE
                ),
                0
            ) * 3600

            +

            COALESCE(
                TRY_CAST(
                    NULLIF(
                        REGEXP_EXTRACT(
                            CAST(PitOutTime AS STRING),
                            'days [0-9]{2}:([0-9]{2}):',
                            1
                        ),
                        ''
                    ) AS DOUBLE
                ),
                0
            ) * 60

            +

            COALESCE(
                TRY_CAST(
                    NULLIF(
                        REGEXP_EXTRACT(
                            CAST(PitOutTime AS STRING),
                            'days [0-9]{2}:[0-9]{2}:([0-9]+[.]?[0-9]*)',
                            1
                        ),
                        ''
                    ) AS DOUBLE
                ),
                0
            )
    END AS pit_out_seconds,
    CASE
        WHEN PitInTime IS NULL
          OR LOWER(TRIM(CAST(PitInTime AS STRING)))
             IN ('', 'nat', 'null', 'none')
        THEN 0

        ELSE
            COALESCE(
                TRY_CAST(
                    NULLIF(
                        REGEXP_EXTRACT(
                            CAST(PitInTime AS STRING),
                            '([0-9]+) days',
                            1
                        ),
                        ''
                    ) AS DOUBLE
                ),
                0
            ) * 86400

            +

            COALESCE(
                TRY_CAST(
                    NULLIF(
                        REGEXP_EXTRACT(
                            CAST(PitInTime AS STRING),
                            'days ([0-9]{2}):',
                            1
                        ),
                        ''
                    ) AS DOUBLE
                ),
                0
            ) * 3600

            +

            COALESCE(
                TRY_CAST(
                    NULLIF(
                        REGEXP_EXTRACT(
                            CAST(PitInTime AS STRING),
                            'days [0-9]{2}:([0-9]{2}):',
                            1
                        ),
                        ''
                    ) AS DOUBLE
                ),
                0
            ) * 60

            +

            COALESCE(
                TRY_CAST(
                    NULLIF(
                        REGEXP_EXTRACT(
                            CAST(PitInTime AS STRING),
                            'days [0-9]{2}:[0-9]{2}:([0-9]+[.]?[0-9]*)',
                            1
                        ),
                        ''
                    ) AS DOUBLE
                ),
                0
            )
    END AS pit_in_seconds
    

FROM formula1.laps
""")

DataFrame[]

In [113]:
spark.sql("""SELECT * FROM laps_clean""").show(5)

+----+--------------+------------+-------------+-------------------+----------+----------+----------------+-----+--------+--------+--------+-----------------+----------+-----------------+---------------+--------------+
|year|session_number|session_type|driver_number|driver_abbreviation| team_name|lap_number|running_position|stint|compound|speed_i1|speed_i2|finish_line_speed|speed_trap| lap_time_seconds|pit_out_seconds|pit_in_seconds|
+----+--------------+------------+-------------+-------------------+----------+----------+----------------+-----+--------+--------+--------+-----------------+----------+-----------------+---------------+--------------+
|2020|            16|           R|           10|                GAS|AlphaTauri|         1|             7.0|    1|    SOFT|   290.0|    86.0|            244.0|     241.0|82.82300000000001|            0.0|           0.0|
|2020|            16|           R|           10|                GAS|AlphaTauri|         2|             7.0|    1|    SOFT|  

In [114]:
spark.sql("""
SELECT
    year,
    session_number,
    session_type,
    driver_number,
    lap_number,
    lap_time_seconds,
    pit_out_seconds,
    pit_in_seconds
FROM laps_clean
WHERE lap_time_seconds IS NOT NULL
ORDER BY year, session_number, driver_number, lap_number
LIMIT 30
""").show()

+----+--------------+------------+-------------+----------+------------------+---------------+--------------+
|year|session_number|session_type|driver_number|lap_number|  lap_time_seconds|pit_out_seconds|pit_in_seconds|
+----+--------------+------------+-------------+----------+------------------+---------------+--------------+
|2018|             1|           Q|           10|         1|             120.4|        467.987|           0.0|
|2018|             1|           Q|           10|         2|            85.295|            0.0|           0.0|
|2018|             1|           Q|           10|         3|           108.298|            0.0|           0.0|
|2018|             1|           Q|           10|         4|            85.425|            0.0|           0.0|
|2018|             1|           Q|           10|         6|           117.672|       1165.265|           0.0|
|2018|             1|           Q|           11|         1|109.61500000000001|        311.666|           0.0|
|2018|    

In [115]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW race_outcomes AS

SELECT
    year,
    session_number,
    driver_number,
    driver_name,
    team_name,

    result_position AS target_position,
    championship_points AS target_championship_points,

    grid_position,
    status,

    CASE
        WHEN grid_position > 0
         AND result_position IS NOT NULL
        THEN grid_position - result_position
        ELSE NULL
    END AS positions_gained,

    CASE
        WHEN status IS NULL THEN NULL
        WHEN LOWER(status) = 'finished' THEN 0
        WHEN LOWER(status) LIKE '+%lap%' THEN 0
        ELSE 1
    END AS dnf_flag

FROM results_clean

WHERE session_type = 'R'
  AND result_position IS NOT NULL
""")

DataFrame[]

In [116]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW qualifying_results AS

WITH qualifying AS (
    SELECT
        year,
        session_number,
        driver_number,
        driver_name,
        team_name,
        result_position AS qualifying_position
    FROM results_clean
    WHERE session_type = 'Q'
      AND result_position IS NOT NULL
),

team_windows AS (
    SELECT
        *,

        COUNT(*) OVER (
            PARTITION BY year, session_number
        ) AS qualifying_field_size,

        COUNT(*) OVER (
            PARTITION BY year, session_number, team_name
        ) AS team_driver_count,

        SUM(qualifying_position) OVER (
            PARTITION BY year, session_number, team_name
        ) AS team_position_sum

    FROM qualifying
)

SELECT
    year,
    session_number,
    driver_number,
    driver_name,
    team_name,

    qualifying_position,
    qualifying_field_size,

    CASE
        WHEN qualifying_field_size > 1
        THEN
            (qualifying_position - 1.0)
            / (qualifying_field_size - 1.0)
    END AS qualifying_position_percentile,

    CASE
        WHEN qualifying_position <= 10 THEN 1
        ELSE 0
    END AS qualified_top_10,

    CASE
        WHEN team_driver_count = 2
        THEN team_position_sum - qualifying_position
    END AS teammate_qualifying_position,

    CASE
        WHEN team_driver_count = 2
        THEN
            qualifying_position
            - (team_position_sum - qualifying_position)
    END AS qualifying_vs_teammate

FROM team_windows
""")

DataFrame[]

In [117]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW qualifying_lap_features AS

SELECT
    year,
    session_number,
    driver_number,

    COUNT(lap_time_seconds) AS valid_qualifying_laps,

    MIN(lap_time_seconds) AS best_qualifying_lap,

    PERCENTILE_APPROX(
        lap_time_seconds,
        0.50
    ) AS median_qualifying_lap,

    AVG(lap_time_seconds) AS mean_qualifying_lap,

    STDDEV_SAMP(
        lap_time_seconds
    ) AS qualifying_lap_sd,

    MAX(speed_i1) AS maximum_speed_i1,
    MAX(speed_i2) AS maximum_speed_i2,
    MAX(finish_line_speed) AS maximum_finish_line_speed,
    MAX(speed_trap) AS maximum_speed_trap,

    COUNT(DISTINCT stint) AS qualifying_stints,
    COUNT(DISTINCT compound) AS qualifying_compounds

FROM laps_clean

WHERE session_type = 'Q'
  AND lap_time_seconds BETWEEN 20 AND 400

GROUP BY
    year,
    session_number,
    driver_number
""")

DataFrame[]

In [118]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW qualifying_features AS

WITH combined AS (
    SELECT
        q.*,

        l.valid_qualifying_laps,
        l.best_qualifying_lap,
        l.median_qualifying_lap,
        l.mean_qualifying_lap,
        l.qualifying_lap_sd,

        l.maximum_speed_i1,
        l.maximum_speed_i2,
        l.maximum_finish_line_speed,
        l.maximum_speed_trap,

        l.qualifying_stints,
        l.qualifying_compounds

    FROM qualifying_results q

    LEFT JOIN qualifying_lap_features l
      ON q.year = l.year
     AND q.session_number = l.session_number
     AND q.driver_number = l.driver_number
)

SELECT
    *,

    best_qualifying_lap
    - MIN(best_qualifying_lap) OVER (
        PARTITION BY year, session_number
    ) AS qualifying_gap_to_pole

FROM combined
""")

DataFrame[]

In [119]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW qualifying_weather AS

SELECT
    CAST(year AS INT) AS year,
    CAST(session_number AS INT) AS session_number,

    AVG(TRY_CAST(AirTemp AS DOUBLE)) AS qualifying_air_temperature,
    AVG(TRY_CAST(TrackTemp AS DOUBLE)) AS qualifying_track_temperature,
    AVG(TRY_CAST(Humidity AS DOUBLE)) AS qualifying_humidity,
    AVG(TRY_CAST(Pressure AS DOUBLE)) AS qualifying_pressure,
    AVG(TRY_CAST(WindSpeed AS DOUBLE)) AS qualifying_wind_speed,

    MAX(
        CASE
            WHEN LOWER(TRIM(CAST(Rainfall AS STRING)))
                 IN ('true', '1', '1.0')
            THEN 1
            ELSE 0
        END
    ) AS qualifying_rain

FROM formula1.weather_data

WHERE UPPER(TRIM(CAST(type_of_race AS STRING))) = 'Q'

GROUP BY
    CAST(year AS INT),
    CAST(session_number AS INT)
""")

DataFrame[]

In [120]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW driver_history AS

SELECT
    r.*,

    LAG(target_position, 1) OVER (
        PARTITION BY driver_name
        ORDER BY year, session_number
    ) AS previous_finish,

    LAG(target_championship_points, 1) OVER (
        PARTITION BY driver_name
        ORDER BY year, session_number
    ) AS previous_points,

    LAG(dnf_flag, 1) OVER (
        PARTITION BY driver_name
        ORDER BY year, session_number
    ) AS previous_dnf,

    AVG(target_position) OVER (
        PARTITION BY driver_name
        ORDER BY year, session_number
        ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
    ) AS average_finish_last_3,

    AVG(target_position) OVER (
        PARTITION BY driver_name
        ORDER BY year, session_number
        ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
    ) AS average_finish_last_5,

    STDDEV_SAMP(target_position) OVER (
        PARTITION BY driver_name
        ORDER BY year, session_number
        ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
    ) AS finish_sd_last_5,

    AVG(COALESCE(target_championship_points, 0.0)) OVER (
        PARTITION BY driver_name
        ORDER BY year, session_number
        ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
    ) AS average_points_last_5,

    SUM(COALESCE(target_championship_points, 0.0)) OVER (
        PARTITION BY driver_name
        ORDER BY year, session_number
        ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
    ) AS total_points_last_5,

    AVG(dnf_flag) OVER (
        PARTITION BY driver_name
        ORDER BY year, session_number
        ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
    ) AS dnf_rate_last_5,

    AVG(positions_gained) OVER (
        PARTITION BY driver_name
        ORDER BY year, session_number
        ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
    ) AS average_positions_gained_last_5,

    SUM(COALESCE(target_championship_points, 0.0)) OVER (
        PARTITION BY driver_name, year
        ORDER BY session_number
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS season_points_before_race,

    COUNT(*) OVER (
        PARTITION BY driver_name
        ORDER BY year, session_number
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS career_races_before

FROM race_outcomes r
""")

DataFrame[]

In [121]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW qualifying_history AS

SELECT
    q.*,

    AVG(qualifying_position) OVER (
        PARTITION BY driver_name
        ORDER BY year, session_number
        ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
    ) AS average_qualifying_position_last_3,

    AVG(qualifying_position) OVER (
        PARTITION BY driver_name
        ORDER BY year, session_number
        ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
    ) AS average_qualifying_position_last_5,

    AVG(qualifying_gap_to_pole) OVER (
        PARTITION BY driver_name
        ORDER BY year, session_number
        ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
    ) AS average_gap_to_pole_last_5

FROM qualifying_features q
""")

DataFrame[]

In [122]:
spark.sql("SELECT * FROM race_outcomes").show(5, truncate=False)

+----+--------------+-------------+---------------+-----------+---------------+--------------------------+-------------+--------+----------------+--------+
|year|session_number|driver_number|driver_name    |team_name  |target_position|target_championship_points|grid_position|status  |positions_gained|dnf_flag|
+----+--------------+-------------+---------------+-----------+---------------+--------------------------+-------------+--------+----------------+--------+
|2025|9             |81           |Oscar Piastri  |McLaren    |1              |25.0                      |1            |Finished|0               |0       |
|2025|9             |4            |Lando Norris   |McLaren    |2              |18.0                      |2            |Finished|0               |0       |
|2025|9             |16           |Charles Leclerc|Ferrari    |3              |15.0                      |7            |Finished|4               |0       |
|2025|9             |63           |George Russell |Mercedes   |4

In [123]:

spark.sql("""
CREATE OR REPLACE TEMP VIEW team_race_results AS

SELECT
    year,
    session_number,
    team_name,

    AVG(target_position) AS team_event_average_finish,

    SUM(
        COALESCE(target_championship_points, 0.0)
    ) AS team_event_points,

    AVG(dnf_flag) AS team_event_dnf_rate

FROM race_outcomes

WHERE team_name IS NOT NULL

GROUP BY
    year,
    session_number,
    team_name
""")

DataFrame[]

In [124]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW race_weather AS

SELECT
    CAST(year AS INT) AS year,
    CAST(session_number AS INT) AS session_number,

    AVG(
        TRY_CAST(AirTemp AS DOUBLE)
    ) AS race_air_temperature,

    AVG(
        TRY_CAST(TrackTemp AS DOUBLE)
    ) AS race_track_temperature,

    AVG(
        TRY_CAST(Humidity AS DOUBLE)
    ) AS race_humidity,

    AVG(
        TRY_CAST(Pressure AS DOUBLE)
    ) AS race_pressure,

    AVG(
        TRY_CAST(WindSpeed AS DOUBLE)
    ) AS race_wind_speed,

    MAX(
        TRY_CAST(WindSpeed AS DOUBLE)
    ) AS race_max_wind_speed,

    MIN(
        TRY_CAST(AirTemp AS DOUBLE)
    ) AS race_min_air_temperature,

    MAX(
        TRY_CAST(AirTemp AS DOUBLE)
    ) AS race_max_air_temperature,

    MAX(
        CASE
            WHEN LOWER(TRIM(CAST(Rainfall AS STRING)))
                 IN ('true', '1', '1.0')
            THEN 1
            ELSE 0
        END
    ) AS race_rain,

    AVG(
        CASE
            WHEN LOWER(TRIM(CAST(Rainfall AS STRING)))
                 IN ('true', '1', '1.0')
            THEN 1.0
            ELSE 0.0
        END
    ) AS race_rain_fraction

FROM formula1.weather_data

WHERE UPPER(TRIM(CAST(type_of_race AS STRING))) = 'R'

GROUP BY
    CAST(year AS INT),
    CAST(session_number AS INT)
""")

DataFrame[]

In [125]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW team_history AS

SELECT
    t.*,

    AVG(team_event_average_finish) OVER (
        PARTITION BY team_name
        ORDER BY year, session_number
        ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
    ) AS team_average_finish_last_3,

    AVG(team_event_average_finish) OVER (
        PARTITION BY team_name
        ORDER BY year, session_number
        ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
    ) AS team_average_finish_last_5,

    AVG(team_event_points) OVER (
        PARTITION BY team_name
        ORDER BY year, session_number
        ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
    ) AS team_average_points_last_5,

    AVG(team_event_dnf_rate) OVER (
        PARTITION BY team_name
        ORDER BY year, session_number
        ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
    ) AS team_dnf_rate_last_5

FROM team_race_results t
""")

DataFrame[]

In [126]:
ratings_text = """
Driver	Rating	Year
Lewis Hamilton	94	22
Max Verstappen	94	22
Charles Leclerc	92	22
George Russell	90	22
Lando Norris	90	22
Fernando Alonso	89	22
Sergio Perez	88	22
Valtteri Bottas	88	22
Carlos Sainz	87	22
Sebastian Vettel	85	22
Pierre Gasly	84	22
Esteban Ocon	83	22
Daniel Ricciardo	83	22
Alex Albon	82	22
Kevin Magnussen	81	22
Lance Stroll	80	22
Yuki Tsuonda	78	22
Mick Schumacher	77	22
Zhou Guanyu	70	22
Nicholas Latifi	70	22
Lewis Hamilton	94	21
Valtteri Bottas	90	21
Max Verstappen	93	21
Sergio Perez	86	21
Charles Leclerc	89	21
Carlos Sainz Jr.	89	21
Sebastian Vettel	89	21
Lance Stroll	82	21
Daniel Ricciardo	89	21
Lando Norris	89	21
Fernando Alonso	89	21
Esteban Ocon	83	21
Pierre Gasly	89	21
Yuki Tsunoda	77	21
Kimi Raikkonen	87	21
Antonio Giovinazzi	81	21
Mick Schumacher	80	21
Nikita Mazepin	67	21
George Russell	81	21
Nicholas Latifi	72	21
Lewis Hamilton	94	20
Valtteri Bottas	90	20
Max Verstappen	90	20
Charles Leclerc	86	20
Sebastian Vettel	89	20
Carlos Sainz Jr.	82	20
Pierre Gasly	80	20
Alexander Albon	79	20
Daniel Ricciardo	87	20
Sergio Perez	85	20
Lando Norris	79	20
Kimi Raikkonen	87	20
Daniil Kvyat	80	20
Lance Stroll	78	20
Kevin Magnussen	78	20
Antonio Giovinazzi	73	20
Romain Grosjean	80	20
George Russell	75	20
Esteban Ocon	80	20
Nicholas Latifi	64	20
Max Verstappen	95	26
Lewis Hamilton	92	26
George Russell	92	26
Charles Leclerc	92	26
Lando Norris	92	26
Oscar Piastri	91	26
Fernando Alonso	89	26
Kimi Antonelli	88	26
Pierre Gasly	85	26
Carlos Sainz	85	26
Nico Hülkenberg	85	26
Sergio Perez	85	26
Isack Hadjar	84	26
Esteban Ocon	84	26
Oliver Bearman	83	26
Alexander Albon	83	26
Valtteri Bottas	82	26
Gabriel Bortoleto	81	26
Liam Lawson	80	26
Lance Stroll	77	26
Franco Colapinto	73	26
Arvid Lindblad	73	26
Max Verstappen	96	24
Fernando Alonso	92	24
Carlos Sainz	89	24
Charles Leclerc	89	24
Lando Norris	89	24
Lewis Hamilton	89	24
George Russell	87	24
Sergio Perez	87	24
Alex Albon	85	24
Oscar Piastri	84	24
Pierre Gasly	84	24
Esteban Ocon	83	24
Daniel Ricciardo	82	24
Nico Hulkenberg	82	24
Valtteri Bottas	81	24
Yuki Tsunoda	81	24
Kevin Magnussen	80	24
Lance Stroll	80	24
Zhou Guanyu	80	24
Logan Sargeant	70	24
Max Verstappen	96	23
Lewis Hamilton	92	23
Fernando Alonso	92	23
Charles Leclerc	90	23
Carlos Sainz Jr.	90	23
Sergio Perez	87	23
George Russell	87	23
Lando Norris	89	23
Valtteri Bottas	87	23
Esteban Ocon	86	23
Pierre Gasly	85	23
Oscar Piastri	85	23
Lance Stroll	84	23
Alexander Albon	83	23
Yuki Tsunoda	83	23
Kevin Magnussen	81	23
Nico Hulkenberg	80	23
Zhou Guanyu	78	23
Logan Sargeant	71	23
Nyck de Vries	71	23
Max Verstappen	95	25
Lando Norris	92	25
Charles Leclerc	92	25
George Russell	92	25
Oscar Piastri	91	25
Lewis Hamilton	91	25
Fernando Alonso	89	25
Nico Hülkenberg	86	25
Alexander Albon	85	25
Pierre Gasly	85	25
Carlos Sainz	85	25
Esteban Ocon	83	25
Isack Hadjar	82	25
Kimi Antonelli	80	25
Oliver Bearman	80	25
Gabriel Bortoleto	79	25
Yuki Tsunoda	77	25
Liam Lawson	77	25
Lance Stroll	77	25
Franco Colapinto	73	25
"""

In [127]:
import pandas as pd
from io import StringIO

ratings_pd = pd.read_csv(
    StringIO(ratings_text.strip()),
    sep="\t"
)

ratings_pd = ratings_pd.rename(
    columns={
        "Driver": "rating_driver_name",
        "Rating": "driver_rating",
        "Year": "year"
    }
)

ratings_pd["year"] = ratings_pd["year"].astype(int)

ratings_pd["year"] = ratings_pd["year"].apply(
    lambda year: year + 2000
    if year < 100
    else year
)

ratings_pd["driver_rating"] = (
    ratings_pd["driver_rating"]
    .astype(float)
)

ratings_pd.head()

,rating_driver_name,driver_rating,year
0,Lewis Hamilton,94.0,2022
1,Max Verstappen,94.0,2022
2,Charles Leclerc,92.0,2022
3,George Russell,90.0,2022
4,Lando Norris,90.0,2022


In [128]:
import re
import unicodedata

name_aliases = {
    "verstappen": "max verstappen",

    "alex albon": "alexander albon",

    "carlos sainz jr": "carlos sainz",

    "zhou guanyu": "guanyu zhou",

    "yuki tsuonda": "yuki tsunoda",

    "kimi antonelli": "andrea kimi antonelli",

    "nico hulkenberg": "nico hulkenberg",
    "kimi raikkonen": "kimi raikkonen"
}


def normalize_driver_name(name):
    if name is None:
        return None

    normalized = unicodedata.normalize(
        "NFKD",
        str(name)
    )

    normalized = "".join(
        character
        for character in normalized
        if not unicodedata.combining(character)
    )

    normalized = normalized.lower()

    normalized = re.sub(
        r"[^a-z0-9 ]",
        " ",
        normalized
    )

    normalized = re.sub(
        r"\s+",
        " ",
        normalized
    ).strip()

    return name_aliases.get(
        normalized,
        normalized
    )

In [129]:
ratings_pd["normalized_driver_name"] = (
    ratings_pd["rating_driver_name"]
    .apply(normalize_driver_name)
)

ratings_pd[
    [
        "year",
        "rating_driver_name",
        "normalized_driver_name",
        "driver_rating"
    ]
].head(30)

ratings_spark = spark.createDataFrame(
    ratings_pd[
        [
            "year",
            "rating_driver_name",
            "normalized_driver_name",
            "driver_rating"
        ]
    ]
)
# i want to make a career average column as well
from pyspark.sql import Window
import pyspark.sql.functions as F

# 1. Define the window partitioned by the driver's name
driver_window = Window.partitionBy("normalized_driver_name")

# 2. Calculate the average over that window
ratings_spark = ratings_spark.withColumn(
    "career_average_rating", 
    F.avg("driver_rating").over(driver_window)
)

ratings_spark.show(30, truncate=False)


+----+------------------+----------------------+-------------+---------------------+
|year|rating_driver_name|normalized_driver_name|driver_rating|career_average_rating|
+----+------------------+----------------------+-------------+---------------------+
|2022|Alex Albon        |alexander albon       |82.0         |82.83333333333333    |
|2020|Alexander Albon   |alexander albon       |79.0         |82.83333333333333    |
|2026|Alexander Albon   |alexander albon       |83.0         |82.83333333333333    |
|2024|Alex Albon        |alexander albon       |85.0         |82.83333333333333    |
|2023|Alexander Albon   |alexander albon       |83.0         |82.83333333333333    |
|2025|Alexander Albon   |alexander albon       |85.0         |82.83333333333333    |
|2026|Kimi Antonelli    |andrea kimi antonelli |88.0         |84.0                 |
|2025|Kimi Antonelli    |andrea kimi antonelli |80.0         |84.0                 |
|2021|Antonio Giovinazzi|antonio giovinazzi    |81.0         |77.

In [130]:
ratings_spark.show(30, truncate=False)


+----+------------------+----------------------+-------------+---------------------+
|year|rating_driver_name|normalized_driver_name|driver_rating|career_average_rating|
+----+------------------+----------------------+-------------+---------------------+
|2022|Alex Albon        |alexander albon       |82.0         |82.83333333333333    |
|2020|Alexander Albon   |alexander albon       |79.0         |82.83333333333333    |
|2026|Alexander Albon   |alexander albon       |83.0         |82.83333333333333    |
|2024|Alex Albon        |alexander albon       |85.0         |82.83333333333333    |
|2023|Alexander Albon   |alexander albon       |83.0         |82.83333333333333    |
|2025|Alexander Albon   |alexander albon       |85.0         |82.83333333333333    |
|2026|Kimi Antonelli    |andrea kimi antonelli |88.0         |84.0                 |
|2025|Kimi Antonelli    |andrea kimi antonelli |80.0         |84.0                 |
|2021|Antonio Giovinazzi|antonio giovinazzi    |81.0         |77.

In [131]:
ratings_spark.createOrReplaceTempView("ratings_table")


In [132]:
#spark.sql(""" SELECT * from team_race_results """).show(5, truncate=False)
#spark.sql(""" SELECT * from qualifying_results """).show(5, truncate=False)
spark.sql(""" SELECT * from laps_clean where session_type = "R" """).show(5, truncate=False)

+----+--------------+------------+-------------+-------------------+----------+----------+----------------+-----+--------+--------+--------+-----------------+----------+-----------------+---------------+--------------+
|year|session_number|session_type|driver_number|driver_abbreviation|team_name |lap_number|running_position|stint|compound|speed_i1|speed_i2|finish_line_speed|speed_trap|lap_time_seconds |pit_out_seconds|pit_in_seconds|
+----+--------------+------------+-------------+-------------------+----------+----------+----------------+-----+--------+--------+--------+-----------------+----------+-----------------+---------------+--------------+
|2020|16            |R           |10           |GAS                |AlphaTauri|1         |7.0             |1    |SOFT    |290.0   |86.0    |244.0            |241.0     |82.82300000000001|0.0            |0.0           |
|2020|16            |R           |10           |GAS                |AlphaTauri|2         |7.0             |1    |SOFT    |15

In [133]:
# ── Split results into race and qualifying classifications ────────────────────
# type_of_race was parsed from the folder name during ingestion ('R' vs 'Q').
# Two events (year, session_number) can share the same DriverNumber, so that's


# ── Overtakes: count every lap-to-lap position improvement during the race ────
# Same logic as f1.ipynb's cell 4, generalized to (year, session_number) instead
# of (year, round) since that's how events are keyed in the database.
from pyspark.sql.window import Window
from pyspark.sql.functions import col, when
from pyspark.sql import functions as F

race_laps = spark.sql(""" SELECT * from laps_clean where session_type = "R" """)

overtake_window = Window.partitionBy("year", "session_number", "driver_number").orderBy("lap_number")

overtakes_spark = (
    race_laps
    .withColumn("prev_position", F.lag("running_position", 1).over(overtake_window))
    .withColumn(
        "overtake_this_lap",
        when(col("prev_position").isNotNull() & (col("running_position") < col("prev_position")), 1).otherwise(0)
    )
    .groupBy("year", "session_number", "driver_number")
    .agg(F.sum("overtake_this_lap").alias("overtakes_made"))
)

overtakes_spark.orderBy(F.desc("overtakes_made")).show(5)


# ── Best pit-stop duration per driver per race ─────────────────────────────────
# Same in-lap / out-lap pairing logic as f1.ipynb's cell 5: an in-lap (where the
# driver entered the pits) is paired with the *next* out-lap (where they left)
# for that same driver in that same race, and the duration is out_time - in_time.
# best_pit_sec is deliberately left null (not filled with 0) when a driver never
# pitted or retired mid-pit-entry — null correctly means "no valid stop to report."
in_laps = (
    race_laps
    .filter(col("pit_in_seconds").isNotNull())
    .select("year", "session_number", "driver_number", "lap_number",
            col("pit_in_seconds").alias("pit_in_time"))
)

out_laps = (
    race_laps
    .filter(col("pit_out_seconds").isNotNull())
    .select("year", "session_number", "driver_number", "lap_number",
            col("pit_out_seconds").alias("pit_out_time"))
)

paired_pits = (
    in_laps.alias("i")
    .join(
        out_laps.alias("o"),
        (col("i.year") == col("o.year")) &
        (col("i.session_number") == col("o.session_number")) &
        (col("i.driver_number") == col("o.driver_number")) &
        (col("o.pit_out_time") > col("i.pit_in_time")),
        how="inner",
    )
    .withColumn("pit_duration_sec", col("o.pit_out_time") - col("i.pit_in_time"))
)


best_pit_spark = (
    paired_pits
    .groupBy(col("i.year").alias("year"), col("i.session_number").alias("session_number"),
              col("i.driver_number").alias("driver_number"))
    .agg(F.min("pit_duration_sec").alias("best_pit_sec"))
)

best_pit_spark.orderBy("year", "session_number", "driver_number").show(5)

# ── Fastest lap flag: whoever set the quickest race lap in each event ─────────
fastest_lap_window = Window.partitionBy("year", "session_number").orderBy("lap_time_seconds")

fastest_lap_spark = (
    race_laps
    .filter(col("lap_time_seconds").isNotNull())
    .withColumn("rnk", F.rank().over(fastest_lap_window))
    .filter(col("rnk") == 1)
    .select("year", "session_number", "driver_number")
    .withColumn("fastest_lap_flag", F.lit(1))
)
fastest_lap_spark.orderBy("year", "session_number", "driver_number").show(5)

# ── Point value constants ──────────────────────────────────────────────────────
### hannah note: took some liberties and assigned points here
# Standard F1-style points-paying positions, same tables as f1.ipynb.
RACE_PTS  = {1: 25, 2: 18, 3: 15, 4: 12, 5: 10, 6: 8, 7: 6, 8: 4, 9: 2, 10: 1}
QUALI_PTS = {1: 10, 2: 9,  3: 8,  4: 7,  5: 6,  6: 5, 7: 4, 8: 3, 9: 2, 10: 1}

# NOTE: f1.ipynb's source cells were cut off before showing the exact multiplier
# used for pts_overtake and pts_fastest. These are set to a sensible default
# (1 point per overtake, 5 points for fastest lap) — change PTS_PER_OVERTAKE /
# FASTEST_LAP_BONUS below.
PTS_PER_OVERTAKE  = 1
FASTEST_LAP_BONUS = 5

def position_to_points(col_name, pts_dict):
    """Map a position column to fantasy points using a lookup dict; unranked = 0."""
    expr = None
    for pos, pts in pts_dict.items():
        expr = when(col(col_name) == pos, pts) if expr is None else expr.when(col(col_name) == pos, pts)
    return expr.otherwise(0)




+----+--------------+-------------+--------------+
|year|session_number|driver_number|overtakes_made|
+----+--------------+-------------+--------------+
|2022|             1|           24|            22|
|2023|             7|           16|            21|
|2023|            17|           11|            21|
|2022|            11|           14|            20|
|2023|            17|           63|            20|
+----+--------------+-------------+--------------+
only showing top 5 rows


+----+--------------+-------------+------------------+
|year|session_number|driver_number|      best_pit_sec|
+----+--------------+-------------+------------------+
|2020|             1|           10|19.798000000000684|
|2020|             1|           11|16.921000000000276|
|2020|             1|           16|18.657999999999447|
|2020|             1|           23|17.129000000000815|
|2020|             1|           26|19.621000000000095|
+----+--------------+-------------+------------------+
only showing top 5 rows


+----+--------------+-------------+----------------+
|year|session_number|driver_number|fastest_lap_flag|
+----+--------------+-------------+----------------+
|2020|             1|            4|               1|
|2020|             2|           55|               1|
|2020|             3|           44|               1|
|2020|             4|           33|               1|
|2020|             5|           44|               1|
+----+--------------+-------------+----------------+
only showing top 5 rows


In [134]:

overtakes_spark.createOrReplaceTempView("overtakes_spark")
best_pit_spark.createOrReplaceTempView("best_pit_spark")
fastest_lap_spark.createOrReplaceTempView("fastest_lap_spark")

modeling_df = spark.sql("""
SELECT
    d.year,
    d.session_number,
    e.Location AS race_location,

    d.driver_number,
    d.driver_name,
    d.team_name,

    q.qualifying_position,
    q.qualifying_field_size,
    q.qualifying_position_percentile,
    q.qualified_top_10,
    q.teammate_qualifying_position,
    q.qualifying_vs_teammate,

    q.valid_qualifying_laps,
    q.best_qualifying_lap,
    q.median_qualifying_lap,
    q.mean_qualifying_lap,
    q.qualifying_lap_sd,
    q.qualifying_gap_to_pole,

    q.maximum_speed_i1,
    q.maximum_speed_i2,
    q.maximum_finish_line_speed,
    q.maximum_speed_trap,

    q.qualifying_stints,
    q.qualifying_compounds,

    q.average_qualifying_position_last_3,
    q.average_qualifying_position_last_5,
    q.average_gap_to_pole_last_5,

    w.race_air_temperature,
    w.race_track_temperature,
    w.race_humidity,
    w.race_pressure,
    w.race_wind_speed,
    w.race_max_wind_speed,
    w.race_min_air_temperature,
    w.race_max_air_temperature,
    w.race_rain,
    w.race_rain_fraction,

    d.previous_finish,
    d.previous_points,
    d.previous_dnf,
    r.driver_rating,

    d.average_finish_last_3,
    d.average_finish_last_5,
    d.finish_sd_last_5,

    d.average_points_last_5,
    d.total_points_last_5,
    d.dnf_rate_last_5,
    d.average_positions_gained_last_5,

    d.season_points_before_race,
    d.career_races_before,

    t.team_average_finish_last_3,
    t.team_average_finish_last_5,
    t.team_average_points_last_5,
    t.team_dnf_rate_last_5,

    d.target_position,
    d.target_championship_points,
    
    o.overtakes_made,
    b.best_pit_sec,
    f.fastest_lap_flag

FROM driver_history d

INNER JOIN qualifying_history q
  ON d.year = q.year
 AND d.session_number = q.session_number
 AND d.driver_number = q.driver_number

LEFT JOIN race_weather w
  ON d.year = w.year
 AND d.session_number = w.session_number

LEFT JOIN team_history t
  ON d.year = t.year
 AND d.session_number = t.session_number
 AND d.team_name = t.team_name

LEFT JOIN event_info e
  ON d.year = e.year
 AND d.session_number = e.session_number

LEFT JOIN ratings_table r
  ON d.year = r.year
 AND d.driver_name = r.rating_driver_name
 
LEFT JOIN overtakes_spark o
  ON d.year = o.year
 AND d.session_number = o.session_number
 AND d.driver_number = o.driver_number

LEFT JOIN best_pit_spark b
  ON d.year = b.year
 AND d.session_number = b.session_number
 AND d.driver_number = b.driver_number

LEFT JOIN fastest_lap_spark f
  ON d.year = f.year
 AND d.session_number = f.session_number
 AND d.driver_number = f.driver_number
""")


    


In [135]:
from pyspark.sql.functions import col, coalesce, lit

modeling_df = modeling_df.withColumn(
    "fantasy_points",
    position_to_points("target_position", RACE_PTS) 
    + position_to_points("qualifying_position", QUALI_PTS) 
    + (col("target_position") - col("qualifying_position")) * 2 
    + coalesce(col("overtakes_made"), lit(0)) * PTS_PER_OVERTAKE 
    + coalesce(col("fastest_lap_flag"), lit(0)) * FASTEST_LAP_BONUS
) 
# set overtakes made and fastest lap flag to 0 if null, then multiply by their respective point values.
# if this doesn't happen the whole value will be displayed as null



In [136]:
modeling_df.show(5, truncate=False)

+----+--------------+-------------+-------------+---------------+---------------+-------------------+---------------------+------------------------------+----------------+----------------------------+----------------------+---------------------+-------------------+---------------------+-------------------+------------------+----------------------+----------------+----------------+-------------------------+------------------+-----------------+--------------------+----------------------------------+----------------------------------+--------------------------+--------------------+----------------------+-----------------+------------------+------------------+-------------------+------------------------+------------------------+---------+------------------+---------------+---------------+------------+-------------+---------------------+---------------------+-----------------+---------------------+-------------------+------------------+-------------------------------+-----------------------

In [137]:
modeling_df.columns

['year',
 'session_number',
 'race_location',
 'driver_number',
 'driver_name',
 'team_name',
 'qualifying_position',
 'qualifying_field_size',
 'qualifying_position_percentile',
 'qualified_top_10',
 'teammate_qualifying_position',
 'qualifying_vs_teammate',
 'valid_qualifying_laps',
 'best_qualifying_lap',
 'median_qualifying_lap',
 'mean_qualifying_lap',
 'qualifying_lap_sd',
 'qualifying_gap_to_pole',
 'maximum_speed_i1',
 'maximum_speed_i2',
 'maximum_finish_line_speed',
 'maximum_speed_trap',
 'qualifying_stints',
 'qualifying_compounds',
 'average_qualifying_position_last_3',
 'average_qualifying_position_last_5',
 'average_gap_to_pole_last_5',
 'race_air_temperature',
 'race_track_temperature',
 'race_humidity',
 'race_pressure',
 'race_wind_speed',
 'race_max_wind_speed',
 'race_min_air_temperature',
 'race_max_air_temperature',
 'race_rain',
 'race_rain_fraction',
 'previous_finish',
 'previous_points',
 'previous_dnf',
 'driver_rating',
 'average_finish_last_3',
 'average_fi

In [138]:
requested_numeric_features = [

    # Current qualifying result
    "qualifying_position",
    "qualifying_field_size",
    "qualifying_position_percentile",
    "qualified_top_10",
    "teammate_qualifying_position",
    "qualifying_vs_teammate",

    # Current qualifying performance
    "valid_qualifying_laps",
    "best_qualifying_lap",
    "median_qualifying_lap",
    "mean_qualifying_lap",
    "qualifying_lap_sd",
    "qualifying_gap_to_pole",

    # Qualifying speed
    "maximum_speed_i1",
    "maximum_speed_i2",
    "maximum_finish_line_speed",
    "maximum_speed_trap",

    # Qualifying session usage
    "qualifying_stints",
    "qualifying_compounds",

    # Historical qualifying form
    "average_qualifying_position_last_3",
    "average_qualifying_position_last_5",
    "average_gap_to_pole_last_5",

    # Actual race weather
    "race_air_temperature",
    "race_track_temperature",
    "race_humidity",
    "race_pressure",
    "race_wind_speed",
    "race_max_wind_speed",
    "race_min_air_temperature",
    "race_max_air_temperature",
    "race_rain",
    "race_rain_fraction",

    # Previous race
    "previous_finish",
    "previous_points",
    "previous_dnf",

    # Historical driver form
    "average_finish_last_3",
    "average_finish_last_5",
    "finish_sd_last_5",
    "average_points_last_5",
    "total_points_last_5",
    "dnf_rate_last_5",
    "average_positions_gained_last_5",
    "driver_rating", 

    # Season and career
    "season_points_before_race",
    "career_races_before",
    "career_average_driver_rating"

    # Historical team performance
    "team_average_finish_last_3",
    "team_average_finish_last_5",
    "team_average_points_last_5",
    "team_dnf_rate_last_5"
    
    #fantasy points columns
    'overtakes_made',
    'best_pit_sec',
    'fastest_lap_flag',
    'fantasy_points'
]

categorical_features = [
    "driver_name",
    "team_name",
    "race_location"
]

In [ ]:
#write the dataframe to csv
modeling_df.write.csv("/Users/alliewandling/Documents/uva_masters/BigData/Formula-1/data/model_master_dataset.csv", header=True)